In [1]:
# =============================================================================
# FIREWATCH AI — Fáze 3: Trénink umělé inteligence
# Autor: Alexandre Basseville
# Prostředí: Google Colab
# Co skript dělá: Vezme vyčištěná data, rozdělí je, naučí na nich Random Forest
#                 a uloží "mozek" (model) do souboru pro pozdější použití.
# =============================================================================

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import joblib # Knihovna pro uložení natrénovaného modelu

print("=" * 60)
print("   FIREWATCH AI — Trénink modelu (Random Forest)")
print("=" * 60)

# 1. NAČTENÍ DAT
# V Colabu máme soubor přímo ve stejné složce, kam jsme ho nahráli
df = pd.read_csv("fire_processed.csv")
print(f" Načteno záznamů k učení: {len(df)}")

# 2. ROZDĚLENÍ NA VSTUPY (X) A VÝSTUP (y)
# X jsou všechny fyzikální veličiny, podle kterých hádáme
# y je náš cíl (Label), který chceme uhodnout (riziko_pozaru)
X = df.drop(columns=["riziko_pozaru"])
y = df["riziko_pozaru"]

# 3. ROZDĚLENÍ NA TRÉNOVACÍ A TESTOVACÍ SADU (80 % / 20 %)
# AI nesmíme testovat na datech, která už viděla! Proto si 20 % schováme bokem.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f" Trénovací data: {len(X_train)} dní (na tomhle se učí)")
print(f" Testovací data: {len(X_test)} dní (na tomhle to otestujeme)")

# 4. VYTVOŘENÍ A TRÉNINK MODELU
# Random Forest (Náhodný les) je skvělý algoritmus, který vytváří spoustu
# rozhodovacích stromů a hledá v datech skrytá pravidla (např. vliv větru a sucha).
model = RandomForestClassifier(n_estimators=100, random_state=42)

print("\n Probíhá učení modelu... (studuje fyziku a počasí)")
model.fit(X_train, y_train) # TADY SE DĚJE TA HLAVNÍ MAGIE!

# 5. TESTOVÁNÍ MODELU A VÝSLEDKY
# Zeptáme se modelu, jak by rozhodl o těch 20 % dat, co neviděl
predikce = model.predict(X_test)

# Spočítáme přesnost (kolikrát se trefil do reálného výsledku)
presnost = accuracy_score(y_test, predikce)
print("\n" + "=" * 60)
print(f" PŘESNOST MODELU: {presnost * 100:.2f} %")
print("=" * 60)

# 6. MATICE ZÁMĚN (Confusion Matrix)
# Ukáže nám, v čem přesně se model spletl a v čem měl pravdu
matice = confusion_matrix(y_test, predikce)

print("\n Detailní výsledky (Matice záměn):")
print(f"  - Správně určil BEZPEČÍ: {matice[0][0]} krát")
print(f"  - Správně určil POŽÁR:   {matice[1][1]} krát")
print(f"  - Falešný poplach (řekl požár, ale bylo bezpečno): {matice[0][1]} krát")
print(f"  - Minul požár (řekl bezpečno, ale byl požár):      {matice[1][0]} krát")

# 7. ULOŽENÍ HOTOVÉHO MOZKU
# Tento soubor pak stáhneme z Colabu do počítače pro Fázi 4 (Aplikaci)
joblib.dump(model, "firewatch_model.pkl")

# Abychom to pak v aplikaci uměli správně naškálovat, uložíme i jména sloupců
joblib.dump(list(X.columns), "firewatch_features.pkl")

print("\n Model byl úspěšně natrénován a uložen jako 'firewatch_model.pkl'!")
print("   (Nezapomeň si ho stáhnout z Colabu do svého PC!)")

   FIREWATCH AI — Trénink modelu (Random Forest)
 Načteno záznamů k učení: 2630
 Trénovací data: 2104 dní (na tomhle se učí)
 Testovací data: 526 dní (na tomhle to otestujeme)

 Probíhá učení modelu... (studuje fyziku a počasí)

 PŘESNOST MODELU: 100.00 %

 Detailní výsledky (Matice záměn):
  - Správně určil BEZPEČÍ: 433 krát
  - Správně určil POŽÁR:   93 krát
  - Falešný poplach (řekl požár, ale bylo bezpečno): 0 krát
  - Minul požár (řekl bezpečno, ale byl požár):      0 krát

 Model byl úspěšně natrénován a uložen jako 'firewatch_model.pkl'!
   (Nezapomeň si ho stáhnout z Colabu do svého PC!)


In [2]:
# =============================================================================
# FIREWATCH AI — Fáze 3: Trénink Neuronové Sítě (Deep Learning) + Validace
# Autor: Alexandre Basseville
# Prostředí: Google Colab
# =============================================================================

import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import joblib

print("=" * 60)
print("   FIREWATCH AI — Trénink Neuronové Sítě (Deep Learning)")
print("=" * 60)

# 1. NAČTENÍ DAT
df = pd.read_csv("fire_processed.csv")
X = df.drop(columns=["riziko_pozaru"])
y = df["riziko_pozaru"]

# 2. VĚDECKÝ DŮKAZ: KŘÍŽOVÁ VALIDACE (Cross-Validation)
print("\n🔬 KROK 1: Provádím 5-Fold Křížovou validaci...")
print("   (Testuji stabilitu modelu na 5 různých kouscích dat, abych vyloučil náhodu)")

# Vytvoříme architekturu neuronové sítě:
# 2 skryté vrstvy (jedna s 16 neurony, druhá s 8 neurony), max 500 iterací učení
nn_architektura = MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=500, random_state=42)

# Spustíme křížovou validaci
skore_validace = cross_val_score(nn_architektura, X, y, cv=5)

print(f"   -> Výsledky 5 testů: {skore_validace}")
print(f"   -> Průměrná garantovaná přesnost: {skore_validace.mean() * 100:.2f} %")

# 3. ROZDĚLENÍ A FINÁLNÍ TRÉNINK
print("\n🧠 KROK 2: Trénuji finální Neuronovou síť...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=500, random_state=42)
model.fit(X_train, y_train) # Trénink synapsí

# 4. TESTOVÁNÍ A MATICE ZÁMĚN
predikce = model.predict(X_test)
presnost = accuracy_score(y_test, predikce)

print("\n" + "=" * 60)
print(f"🎯 FINÁLNÍ PŘESNOST SÍTĚ: {presnost * 100:.2f} %")
print("=" * 60)

matice = confusion_matrix(y_test, predikce)
print("\n📊 Matice záměn (Confusion Matrix):")
print(f"  - Správně BEZPEČÍ: {matice[0][0]}")
print(f"  - Správně POŽÁR:   {matice[1][1]}")
print(f"  - Falešný poplach: {matice[0][1]}")
print(f"  - Minul požár:     {matice[1][0]}")

# 5. ULOŽENÍ MOZKU
joblib.dump(model, "firewatch_model.pkl")
joblib.dump(list(X.columns), "firewatch_features.pkl")

print("\n💾 Neuronová síť uložena do 'firewatch_model.pkl'!")

   FIREWATCH AI — Trénink Neuronové Sítě (Deep Learning)

🔬 KROK 1: Provádím 5-Fold Křížovou validaci...
   (Testuji stabilitu modelu na 5 různých kouscích dat, abych vyloučil náhodu)


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


   -> Výsledky 5 testů: [0.98859316 0.9391635  0.98859316 0.97908745 0.98669202]
   -> Průměrná garantovaná přesnost: 97.64 %

🧠 KROK 2: Trénuji finální Neuronovou síť...

🎯 FINÁLNÍ PŘESNOST SÍTĚ: 98.48 %

📊 Matice záměn (Confusion Matrix):
  - Správně BEZPEČÍ: 427
  - Správně POŽÁR:   91
  - Falešný poplach: 6
  - Minul požár:     2

💾 Neuronová síť uložena do 'firewatch_model.pkl'!


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [3]:
# =============================================================================
# FIREWATCH AI — Fáze 3: Trénink Neuronové Sítě (Deep Learning) + Validace
# Autor: Alexandre Basseville
# Prostředí: Google Colab
# =============================================================================

import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.inspection import permutation_importance # Knihovna pro XAI (vysvětlení černé skříňky)
import joblib

print("=" * 60)
print("   FIREWATCH AI — Trénink Neuronové Sítě (Deep Learning)")
print("=" * 60)

# 1. NAČTENÍ DAT
df = pd.read_csv("fire_processed.csv")
X = df.drop(columns=["riziko_pozaru"])
y = df["riziko_pozaru"]

# 2. VĚDECKÝ DŮKAZ: KŘÍŽOVÁ VALIDACE (Cross-Validation)
print("\n🔬 KROK 1: Provádím 5-Fold Křížovou validaci...")
print("   (Testuji stabilitu modelu na 5 různých kouscích dat, abych vyloučil náhodu)")

# Vytvoříme architekturu neuronové sítě:
# 2 skryté vrstvy (jedna s 16 neurony, druhá s 8 neurony), max 500 iterací učení
nn_architektura = MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=500, random_state=42)

# Spustíme křížovou validaci
skore_validace = cross_val_score(nn_architektura, X, y, cv=5)

print(f"   -> Výsledky 5 testů: {skore_validace}")
print(f"   -> Průměrná garantovaná přesnost: {skore_validace.mean() * 100:.2f} %")

# 3. ROZDĚLENÍ A FINÁLNÍ TRÉNINK
print("\n🧠 KROK 2: Trénuji finální Neuronovou síť...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=500, random_state=42)
model.fit(X_train, y_train) # Trénink synapsí

# 4. TESTOVÁNÍ A MATICE ZÁMĚN
predikce = model.predict(X_test)
presnost = accuracy_score(y_test, predikce)

print("\n" + "=" * 60)
print(f"🎯 FINÁLNÍ PŘESNOST SÍTĚ: {presnost * 100:.2f} %")
print("=" * 60)

matice = confusion_matrix(y_test, predikce)
print("\n📊 Matice záměn (Confusion Matrix):")
print(f"  - Správně BEZPEČÍ: {matice[0][0]}")
print(f"  - Správně POŽÁR:   {matice[1][1]}")
print(f"  - Falešný poplach: {matice[0][1]}")
print(f"  - Minul požár:     {matice[1][0]}")

# 5. XAI - EXPLAINABLE AI (Zjištění vlivu jednotlivých faktorů)
print("\n🔍 KROK 3: Analyzuji chování sítě pomocí Permutation Importance (XAI)...")
# Neuronová síť je černá skříňka. Tato metoda jí postupně "rozbíjí" jednotlivé vstupy
# a sleduje, jak moc klesne přesnost. Co způsobí největší pád přesnosti, to bylo nejdůležitější.
vysledky_xai = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42)
importances = vysledky_xai.importances_mean
print("   -> Vliv faktorů úspěšně vypočítán!")

# 6. ULOŽENÍ MOZKU A DAT PRO APLIKACI
joblib.dump(model, "firewatch_model.pkl")
joblib.dump(list(X.columns), "firewatch_features.pkl")
joblib.dump(importances, "firewatch_importances.pkl") # Uložíme i naši XAI analýzu pro graf

print("\n💾 Uloženo! Stáhni si všechny tři .pkl soubory k sobě do aplikace!")

   FIREWATCH AI — Trénink Neuronové Sítě (Deep Learning)

🔬 KROK 1: Provádím 5-Fold Křížovou validaci...
   (Testuji stabilitu modelu na 5 různých kouscích dat, abych vyloučil náhodu)


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


   -> Výsledky 5 testů: [0.98859316 0.9391635  0.98859316 0.97908745 0.98669202]
   -> Průměrná garantovaná přesnost: 97.64 %

🧠 KROK 2: Trénuji finální Neuronovou síť...

🎯 FINÁLNÍ PŘESNOST SÍTĚ: 98.48 %

📊 Matice záměn (Confusion Matrix):
  - Správně BEZPEČÍ: 427
  - Správně POŽÁR:   91
  - Falešný poplach: 6
  - Minul požár:     2

🔍 KROK 3: Analyzuji chování sítě pomocí Permutation Importance (XAI)...
   -> Vliv faktorů úspěšně vypočítán!

💾 Uloženo! Stáhni si všechny tři .pkl soubory k sobě do aplikace!


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
